# 04 Edge Pruning KL vs Two-Label Objective

Compare finalized Edge-Pruning circuits trained with KL full-vocab loss against circuits trained with the two-label objective. The notebook loads finalized artifacts by default and can regenerate them with the configured budgets.

In [ ]:
from pathlib import Path
import sys
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from circuit_discovery.run import (
    get_compute_device,
    load_configs,
    load_model,
    load_task_dataset_from_config,
    load_circuit_map,
    evaluation_rows,
    pairwise_iou_rows,
    train_loader_from_config,
)

configs = load_configs()
print("project root:", PROJECT_ROOT)
print("device:", get_compute_device())

params = configs["notebooks"]["04_edge_pruning_kl_vs_ce"]["hyperparams"]
artifacts = configs["artifacts"]["edge_pruning"]["circuits"]
params


In [ ]:
# Optional regeneration. Expensive; leave disabled when browsing saved artifacts.
RUN_EXPERIMENT = False

if RUN_EXPERIMENT:
    from circuit_discovery.algorithms.edge_pruning import EdgePruning, EdgePruningConfig
    from circuit_discovery.utils import set_seed

    output_dir = PROJECT_ROOT / configs["artifacts"]["edge_pruning"]["root"]
    output_dir.mkdir(parents=True, exist_ok=True)
    common = params["common"]
    data = load_task_dataset_from_config(common)

    for objective in ["kl", "two_label"]:
        objective_params = params[objective]
        for seed in common["seeds"]:
            set_seed(seed)
            model = load_model(common["model_name"])
            config = EdgePruningConfig(
                model_name=common["model_name"],
                n_epochs=common["n_epochs"],
                max_steps=common["max_steps"],
                batch_size=common["batch_size"],
                edge_learning_rate=common["edge_learning_rate"],
                reg_edge_learning_rate=common["reg_edge_learning_rate"],
                lr_warmup_steps=common["lr_warmup_steps"],
                edge_logit_init_mean=common["edge_logit_init_mean"],
                edge_logit_init_std=common["edge_logit_init_std"],
                target_edge_sparsity=objective_params["target_edge_sparsity"],
                n_sparsity_warmup_steps=common["n_sparsity_warmup_steps"],
                objective=objective_params["objective"],
                cache_teacher_logits=objective_params["cache_teacher_logits"],
                teacher_cache_dtype=torch.float16,
                seed=seed,
                tqdm_disabled=False,
            )
            runner = EdgePruning(model=model, config=config, device=get_compute_device())
            result = runner.fit(train_loader_from_config(data.train.dataset, common))
            circuit = result.circuit_for_sparsity_budget(
                objective_params["target_edge_sparsity"],
                model=model,
                finalize=True,
            )
            torch.save(
                {
                    "circuit": circuit,
                    "algorithm": "edge_pruning",
                    "objective": objective,
                    "seed": seed,
                },
                output_dir / f"{objective}_seed_{seed}.pt",
            )


In [ ]:
model = load_model(params["common"]["model_name"])
data = load_task_dataset_from_config(params["common"])
for objective, path_map in artifacts.items():
    print(f"\nobjective: {objective}")
    circuits = load_circuit_map(path_map)
    display(evaluation_rows(model, data.test, circuits))
    display(pairwise_iou_rows(circuits))
